#BASE CODE (HARI)

In [ ]:
!pip install -U classiq
!pip install keyrings.alt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import keyring
from keyrings.alt.file import PlaintextKeyring
keyring.set_keyring(PlaintextKeyring())

In [ ]:
import classiq
classiq.authenticate(overwrite=True)

If a browser doesn't automatically open, please visit this URL from any trusted device to authenticate: https://auth.classiq.io/authorize?client_id=f6721qMOVoDAOVkzrv8YaWassRKSFX6Y&response_type=code&audience=https%3A%2F%2Fcadmium-be&redirect_uri=https%3A%2F%2Fauth.classiq.io%2Factivate%3Fuser_code%3DBPFS-HDXM&scope=offline_access
Your user code: BPFS-HDXM


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sc
from tabulate import tabulate

from classiq import *

#General Experiment to generate a big table

In [58]:
import numpy as np
import pandas as pd

# Parameter grid
omegas   = [1]
y0s      = [0,1]
vy0s     = [0,1]
bxs      = [0]
bys      = [0]
k_values = [7]
t_range  = 10

rows = []

for omega in omegas:
    for y0 in y0s:
        for vy0 in vy0s:
            for bx in bxs:
                for by in bys:
                    for k in k_values:
                        # Inputs
                        n = 2  # dimension of vectors x(0) and b
                        t = 1
                        M = np.array([[0, 1], [-1, 0]])
                        x_0 = [1, 1]
                        b = [0, 0]

                        # Constants
                        k = 7
                        x_0_norm = np.linalg.norm(x_0)
                        b_norm = np.linalg.norm(b)
                        M_norm = np.linalg.norm(M)
                        # A = np.multiply(1/np.linalg.norm(M), M)
                        A = M

                        vs1 = []
                        c = 0


                        def VS1(t):
                            global vs1
                            global c
                            c = 0
                            c_m = np.zeros(k + 1)
                            m_factorial = 1
                            for i in range(k + 1):
                                c_m[i] = (x_0_norm * (pow(t, i))) / m_factorial
                                # c_m[i] = (x_0_norm * (pow(t * M_Norm, i))) / m_factorial
                                c += c_m[i]
                                m_factorial *= i + 1

                            c = np.sqrt(c)

                            if t == 0:
                                vs1 = np.eye(k + 1, k + 1)
                                return

                            # Construct Unitary matrix with the first column as defined above in the markdown
                            e = np.zeros(k + 1)
                            e[0] = 1
                            w = np.subtract([np.sqrt(c_m[i]) / c for i in range(k + 1)], e)
                            vs1 = np.subtract(
                                np.identity(k + 1), np.multiply(2 * (1 / np.inner(w, w)), np.outer(w, w))
                            )
                        vs2 = []
                        d = 0


                        def VS2():
                            global vs2
                            global d
                            d = 0
                            d_m = np.zeros(k + 1)
                            n_factorial = 1
                            for i in range(1, k + 1):
                                d_m[i - 1] = (b_norm * (pow(t * M_norm, i - 1))) / n_factorial
                                d += d_m[i - 1]
                                n_factorial *= i + 1
                            d_m[k] = 0
                            d = np.sqrt(d)

                            if d == 0:
                                vs2 = np.eye(k + 1, k + 1)
                                return
                            # Construct Unitary matrix with the first column as defined above in the markdown
                            e = np.zeros(k + 1)
                            e[0] = 1
                            w = np.subtract([np.sqrt(d_m[i]) / d for i in range(k + 1)], e)
                            vs2 = np.subtract(
                                np.identity(k + 1), np.multiply(2 * (1 / np.inner(w, w)), np.outer(w, w))
                            )
                        v = []
                        N = 0


                        def V():
                            global v
                            global N
                            v = []
                            N = np.sqrt(c * c + d * d)
                            if N == 0:
                                v = np.eye(2, 2)
                            else:
                                v.append([c / N, d / N])
                                v.append([d / N, -c / N])
                                v = np.array(v)
                        @qfunc
                        def encoding(x: QNum, ancilla: QNum, y: QBit, t: float):
                            prob_x_0 = []
                            for i in x_0:
                                prob_x_0.append(i / x_0_norm)
                            inplace_prepare_amplitudes(prob_x_0, 0.01, x)

                            VS1(t)
                            VS2()
                            V()

                            unitary(v, y)
                            control(y == 0, lambda: unitary(vs1, ancilla), lambda: unitary(vs2, ancilla))

                        @qfunc
                        def evolution(x: QNum, ancilla: QNum):
                            u_m = np.array([[1, 0], [0, 1]])

                            for i in range(k + 1):
                                U = u_m.copy()
                                control(ancilla == i, lambda U=U: unitary(U, x))
                                u_m = u_m @ A

                        @qfunc
                        def decoding(ancilla: QNum, y: QBit):
                            ws1 = vs1.conj().T
                            ws2 = vs2.conj().T
                            w = v.conj().T
                            control(y == 0, lambda: unitary(ws1, ancilla), lambda: unitary(ws2, ancilla))
                            unitary(w, y)

                        execution_preferences = ExecutionPreferences(
                            num_shots=1,
                            backend_preferences=ClassiqBackendPreferences(
                                backend_name=ClassiqSimulatorBackendNames.SIMULATOR_STATEVECTOR
                            ),
                        )

                        y      = []
                        y_dash = []

                        print(f"Running: omega={omega}, y0={y0}, vy0={vy0}, k={k}")

                        for i in range(t_range + 1):
                            t = i / 10

                            try:
                                qmod = create_model(create_main_for_t(t))
                                qmod = set_execution_preferences(qmod, execution_preferences)
                                qprog = synthesize(qmod)
                                job = execute(qprog)
                                results = job.result_value()

                                for j in results.parsed_state_vector:
                                    if int(j.bitstring[:-dim], 2) == 0:
                                        if int(j.bitstring, 2) == 0:
                                            y.append(np.linalg.norm(j.amplitude) * (N * N))
                                        if int(j.bitstring, 2) == 1:
                                            y_dash.append(np.linalg.norm(j.amplitude) * (N * N))

                            except Exception as e:
                                print(f"  Failed at t={t}: {e}")
                                y.append(np.nan)
                                y_dash.append(np.nan)

                        # Expected energies — generalized analytical solution
                        t_values        = [i / 10 for i in range(t_range + 1)]
                        y_values        = [y0 * np.cos(omega * t) + (vy0 / omega) * np.sin(omega * t) for t in t_values]
                        ydash_values    = [-y0 * omega * np.sin(omega * t) + vy0 * np.cos(omega * t) for t in t_values]

                        kinetic_expected  = [(val**2) / 2 for val in y_values]
                        potential_expected = [(val**2) / 2 for val in ydash_values]
                        kinetic_actual    = [(y_val**2) / 2 for y_val in y]
                        potential_actual  = [(ydash_val**2) / 2 for ydash_val in y_dash]

                        error_bound_kinetic   = [100 * np.abs(a - e) / (a + 1e-12) for a, e in zip(kinetic_expected, kinetic_actual)]
                        error_bound_potential = [100 * np.abs(a - e) / (a + 1e-12) for a, e in zip(potential_expected, potential_actual)]

                        for idx, t in enumerate(t_values):
                            rows.append([
                                omega, y0, vy0, bx, by, k, t,
                                kinetic_expected[idx],
                                kinetic_actual[idx],
                                potential_expected[idx],
                                potential_actual[idx],
                                100 - error_bound_kinetic[idx],
                                100 - error_bound_potential[idx],
                            ])

# Save
columns = [
    "omega", "y0", "vy0", "bx", "by", "k", "t",
    "kinetic_expected", "kinetic_actual",
    "potential_expected", "potential_actual",
    "kinetic_accuracy", "potential_accuracy"
]

data = np.array(rows)
save_path = "/content/drive/MyDrive/harmonic_oscillator/"
import os
os.makedirs(save_path, exist_ok=True)

np.save(save_path + "sweep_results.npy", data)
np.save(save_path + "sweep_columns.npy", np.array(columns))



df = pd.DataFrame(data, columns=columns)
df.to_csv(save_path + "sweep_results.csv", index=False)

print(f"Saved to {save_path}")

Running: omega=1.0, y0=0, vy0=0, k=7
Running: omega=1.0, y0=0, vy0=1, k=7
Running: omega=1.0, y0=1, vy0=0, k=7
Running: omega=1.0, y0=1, vy0=1, k=7
Saved to /content/drive/MyDrive/harmonic_oscillator/
